# 5.04 — Albert quantification of SRR3997504 (per-chromosome L1 aggregation)

Port of legacy `4.02-ach-albert_quantify_SRR3997504` into the R/ggplot reproducibility
scheme. **Stage A** (`.trap/scripts/aggregate_srr_by_chr.py srr3997504`, run on Grace)
produces the tidy `results/srr3997504/aggregate_by_chr.csv`
(`sample, figure, chrom, method, class, value`). **Stage B** (this notebook) renders the
four by-chromosome figures to `reports/figures/srr3997504/`:

- **`SRR3997504_agreggate_mlvs_bowtie_by_chr`** — ML (Albert scores) vs Bowtie2 /
  RepeatMasker, unfiltered.
- **`SRR3997504_filtered_agreggate_mlvs_star_by_chr`** — ML (P(NEGATIVE) < 0.3) vs STAR of
  filtered reads vs STAR of all reads.
- **`SRR3997504_filtered_agreggate_l1em_mlvs_star_by_chr`** — adds L1EM (L1EM.400 intervals
  on the filtered STAR alignment).
- **`SRR3997504_filtered_agreggate_l1em_bwa_by_chr`** — ML vs L1EM re-aligned with BWA,
  aggregated by the L1EM locus chromosome.

Classes: L1HS / L1PA2 / L1PA / LINE/L1. Same data, axes, methods, and per-chromosome
ordering as the legacy PNGs; verify against `notebooks/images/SRR3997504_*_by_chr.png`.


In [ ]:
## Reproducibility & environment capture -------------------------------
set.seed(3469)
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(jsonlite)
})
options(repr.plot.width = 18, repr.plot.height = 9)
theme_set(theme_bw(base_size = 24))


In [ ]:
## Paths, provenance, palette ------------------------------------------
results_dir <- file.path("..", "results", "srr3997504")
figures_dir <- file.path("..", "reports", "figures", "srr3997504")
dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)
cat("results_dir:", results_dir, "\n")
cat("figures_dir:", figures_dir, "\n")

manifest_path <- file.path(results_dir, "manifest.json")
if (file.exists(manifest_path)) {
  manifest <- fromJSON(manifest_path)
  cat("\n-- compute manifest --\n")
  cat("git_commit:", manifest$git_commit, "\n")
  cat("timestamp: ", manifest$timestamp_utc, "\n")
  cat("seed:      ", manifest$seed, "\n")
  if (!is.null(manifest$seed)) stopifnot(manifest$seed == 3469)
} else {
  warning("manifest.json not found — run .trap/scripts/aggregate_srr_by_chr.py first.")
}

# Categorical fills are keyed by subfamily class (fixed palette; red #D62728 stays reserved
# for reference lines per the reproducibility scheme). The quantification METHOD is encoded
# by bar shade (alpha), reproducing the legacy solid/hatched/open encoding without matplotlib
# hatches. Standard chromosome order for the x axis.
class_palette <- c("L1HS" = "#1F77B4", "L1PA2" = "#FF7F0E", "L1PA" = "#2CA02C",
                   "LINE/L1" = "#9467BD", "NEGATIVE" = "#7F7F7F")
class_levels  <- names(class_palette)
chrom_levels  <- c(paste0("chr", 1:22), "chrX", "chrY", "chrMT", "chrM")

save_fig <- function(plot, stem, width = 16, height = 8) {
  for (ext in c("png", "pdf")) {
    f <- file.path(figures_dir, paste0(stem, ".", ext))
    suppressMessages(ggsave(f, plot, width = width, height = height, dpi = 300, bg = "white"))
  }
  invisible(plot)
}


In [ ]:
## Load the tidy table + validation checks ----------------------------
agg_path <- file.path(results_dir, "aggregate_by_chr.csv")
stopifnot("aggregate_by_chr.csv not found — run the Stage-A translator" = file.exists(agg_path))
agg <- fread(agg_path)

req <- c("sample", "figure", "chrom", "method", "class", "value")
stopifnot("missing required columns" = all(req %in% names(agg)))
stopifnot("value must be finite and >= 0" = all(is.finite(agg$value) & agg$value >= 0))

std_chrom <- c(paste0("chr", 1:22), "chrX", "chrY", "chrMT", "chrM")
bad_chrom <- setdiff(unique(agg$chrom), std_chrom)
if (length(bad_chrom) > 0)
  warning(sprintf("non-standard chrom(s) present: %s", paste(bad_chrom, collapse = ", ")))

cat("Loaded", nrow(agg), "rows; figures:\n")
print(agg[, .N, by = figure])


In [ ]:
## Grouped-bar helper (per chromosome, dodged by method x class) --------
# One figure = one `figure` value in the tidy table. Bars are dodged per chromosome;
# fill = subfamily class, alpha = method (primary method solid, others progressively
# fainter). Missing method/class combinations simply produce no bar. A whole method being
# absent -> warning (partial re-runs still render what is present).
plot_by_chr <- function(dt, stem, method_levels, width = 16, height = 8) {
  d <- dt[figure == stem]
  if (nrow(d) == 0) { warning(sprintf("%s: no rows — skipped", stem)); return(invisible(NULL)) }
  present <- intersect(method_levels, unique(d$method))
  if (length(present) < length(method_levels))
    warning(sprintf("%s: missing method(s): %s", stem,
                    paste(setdiff(method_levels, present), collapse = ", ")))

  d <- copy(d)
  d[, method := factor(method, levels = method_levels)]
  d[, class  := factor(class,  levels = class_levels)]
  d <- d[!is.na(method) & !is.na(class)]
  d[, chrom  := factor(chrom, levels = chrom_levels)]
  d <- d[!is.na(chrom)]
  # dodge order: class-major, method-minor (matches the legacy column ordering).
  d[, series := interaction(class, method, drop = TRUE, lex.order = TRUE)]

  alpha_vals <- setNames(seq(1.0, 0.35, length.out = length(method_levels)), method_levels)

  p <- ggplot(d, aes(x = chrom, y = value, fill = class, alpha = method, group = series)) +
    geom_col(position = position_dodge2(preserve = "single", padding = 0.1),
             colour = "grey25", linewidth = 0.12) +
    scale_fill_manual(values = class_palette, drop = TRUE, name = "Class") +
    scale_alpha_manual(values = alpha_vals, drop = TRUE, name = "Method") +
    scale_y_continuous(expand = expansion(mult = c(0, 0.05))) +
    labs(x = NULL, y = "Aggregated L1 reads") +
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom", legend.box = "vertical",
          legend.text = element_text(size = 16), legend.title = element_text(size = 18)) +
    guides(fill = guide_legend(order = 1, nrow = 1, override.aes = list(alpha = 1)),
           alpha = guide_legend(order = 2, nrow = 1))
  save_fig(p, stem, width, height)
  p
}


In [ ]:
## ML (Albert) vs Bowtie2/RepeatMasker (unfiltered)
plot_by_chr(agg, "SRR3997504_agreggate_mlvs_bowtie_by_chr", method_levels = c("ML", "Bowtie2"))


In [ ]:
## ML (filtered) vs STAR filtered vs STAR whole
plot_by_chr(agg, "SRR3997504_filtered_agreggate_mlvs_star_by_chr", method_levels = c("ML", "FILTERED_STAR", "WHOLE_STAR"))


In [ ]:
## + L1EM (L1EM.400 intervals on filtered STAR)
plot_by_chr(agg, "SRR3997504_filtered_agreggate_l1em_mlvs_star_by_chr", method_levels = c("ML", "FILTERED_STAR", "WHOLE_STAR", "L1EM_STAR"))


In [ ]:
## ML vs L1EM (BWA re-alignment), aggregated by L1EM locus chromosome
plot_by_chr(agg, "SRR3997504_filtered_agreggate_l1em_bwa_by_chr", method_levels = c("ML", "L1EM_BWA"))
